# Day 16 — Why divide by $\sqrt{d}$

**Milestone 2:** the mysterious $\sqrt{d}$ in attention. It's not arbitrary — dot products **grow with dimension**, and without rescaling, softmax saturates and learning stalls.

## 1. Review (~5 min)

Recall **day 11** (the score matrix $QK^\top$) and **day 14** (stable softmax).

In [ ]:
import numpy as np
rng = np.random.default_rng(16)

**R1.** (day 11) the score matrix $QK^\top$.

In [ ]:
def r_score_matrix(Q, K):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    Q = rng.normal(size=(3, 4)); K = rng.normal(size=(5, 4))
    assert np.allclose(np.asarray(fn(Q, K)), Q @ K.T)
    print("✅ Review R1 passed")

check_r1(r_score_matrix)

**R2.** (day 14) stable softmax.

In [ ]:
def r_softmax_stable(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    s = rng.normal(size=6); e = np.exp(s - s.max())
    assert np.allclose(np.asarray(fn(s)), e / e.sum())
    print("✅ Review R2 passed")

check_r2(r_softmax_stable)

## 2. Lecture note (~10 min): keeping scores in a sane range

**The itch.** A score is a dot product $q\cdot k = \sum_{i=1}^{d} q_i k_i$. If $q,k$ have independent entries with variance 1, each term has variance 1 and there are $d$ of them, so $\operatorname{Var}(q\cdot k)\approx d$ — the scores spread out like $\sqrt{d}$. In a big model ($d=64,128,\dots$) raw scores get **large**.

**The picture.** Feed large-magnitude scores into softmax and (day 13) it behaves like argmax: one weight $\approx 1$, the rest $\approx 0$ — **saturated**. A saturated softmax is nearly flat in its gradient, so almost nothing learns.

**The fix.** Divide the scores by $\sqrt{d}$:
$$\text{scaled scores} = \frac{QK^\top}{\sqrt{d}}.$$
This pulls the variance back to $\approx 1$ regardless of $d$, keeping softmax in its responsive, *soft* regime.

**Knobs & effect.** $\sqrt{d}$ is the right factor precisely because the standard deviation grows like $\sqrt{d}$. Too-large scores → saturated softmax → vanishing gradients; scaling cures it. (This is the only reason the constant is there.)

**In the wild.** The "scaled" in **scaled dot-product attention** (Vaswani et al. 2017, *Attention Is All You Need*, §3.2.1).

## 3. Exercises (~15 min)

### Exercise 1 — scaled scores
Given `Q` `(n_q, d)` and `K` `(n_k, d)`, return $QK^\top/\sqrt{d}$ (shape `(n_q, n_k)`).

In [ ]:
def scaled_scores(Q, K):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    nq, nk, d = 3, 5, 7
    Q = rng.normal(size=(nq, d)); K = rng.normal(size=(nk, d))
    out = np.asarray(fn(Q, K)); assert out.shape == (nq, nk)
    assert np.allclose(out, Q @ K.T / np.sqrt(d))
    print("✅ Exercise 1 passed")

check_e1(scaled_scores)

### Exercise 2 — variance grows with $d$ (the effect)
Draw `n` pairs of random standard-normal vectors in $\mathbb{R}^d$ and return the **sample variance** of their dot products. Confirm it's $\approx d$ — the reason scores explode in high dimensions.

In [ ]:
def sample_dot_variance(gen, d, n):
    # gen is a numpy Generator; return the sample variance of n random dot products in R^d
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    for d in [4, 16, 64]:
        v = fn(np.random.default_rng(123), d, 40000)
        assert abs(v - d) < 0.15 * d, f"variance {v} not ~ {d}"
    print("✅ Exercise 2 passed")

check_e2(sample_dot_variance)

### Exercise 3 — scaling un-saturates softmax (the effect)
For a score vector `scores` and dimension `d`, return `(max_weight_unscaled, max_weight_scaled)` — the largest softmax weight of `scores` vs. of `scores/√d`. On large scores the unscaled softmax is far peakier (closer to 1).

In [ ]:
def saturation(scores, d):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    scores = np.array([8.0, 9.0, 10.0, 11.0]); d = 64
    mu, ms = fn(scores, d)
    assert mu > ms, "unscaled softmax should be more saturated (peakier)"
    assert mu > 0.5 and ms < 0.4
    print("✅ Exercise 3 passed")

check_e3(saturation)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_score_matrix(Q, K):
    return Q @ K.T

def ref_softmax_stable(s):
    e = np.exp(s - np.max(s)); return e / e.sum()

def ref_scaled_scores(Q, K):
    return Q @ K.T / np.sqrt(Q.shape[1])

def ref_sample_dot_variance(gen, d, n):
    a = gen.normal(size=(n, d)); b = gen.normal(size=(n, d))
    return np.var(np.sum(a * b, axis=1))

def ref_saturation(scores, d):
    return ref_softmax_stable(scores).max(), ref_softmax_stable(scores / np.sqrt(d)).max()

check_r1(ref_score_matrix)
check_r2(ref_softmax_stable)
check_e1(ref_scaled_scores)
check_e2(ref_sample_dot_variance)
check_e3(ref_saturation)
print("\nAll reference solutions pass. 🎉  See you on day 17!")